# PathRank on a Colab GPU

Runs the whole project on a Colab GPU: training on CUDA, then a GPU inference benchmark
(PyTorch, ONNX Runtime CUDA, ONNX Runtime + TensorRT, native TensorRT FP16).

**Before you start:** Runtime → Change runtime type → **A100 GPU** (paid plan). If Colab gives you
an L4 or T4 instead, everything still runs; the results record the GPU you actually got.

What does **not** run in Colab: Triton Inference Server (needs Docker). For Triton numbers use a
GPU VM (RunPod / Vast.ai with the `nvcr.io/nvidia/tritonserver` image, or `scripts/gpu_vm.sh` on GCP)
and `scripts/triton_perf.sh`.

Rough time on an A100: smoke test ~5 min, download ~5 min, full CareerBuilder run 30-60 min.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU, "| torch", torch.__version__, "| CUDA", torch.version.cuda)
if "A100" not in GPU:
    print(f"Note: you got {GPU}, not an A100. Results will say {GPU}; don't label them A100.")

## 2. Get the code
Upload `pathrank.zip` when prompted (or clone your GitHub repo instead).
Optional: mount Google Drive so results survive a disconnect.

In [ ]:
import os, zipfile
from google.colab import files
if not os.path.exists("/content/pathrank"):
    # Option A: upload the zip
    up = files.upload()                       # pick pathrank.zip
    zipfile.ZipFile(next(iter(up))).extractall("/content")
    # Option B instead: !git clone https://github.com/<you>/pathrank /content/pathrank
%cd /content/pathrank
!ls

SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

## 3. Install
Colab's PyTorch is kept as-is. `onnxruntime` (CPU) is swapped for `onnxruntime-gpu`, and TensorRT
comes from NVIDIA's pip index (pulled automatically by the `tensorrt` package).
If the last line says to restart: Runtime → Restart session, then continue from step 4.

In [ ]:
import numpy, importlib
np_before = numpy.__version__
!pip install -q -r requirements.txt kaggle
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu tensorrt
import subprocess, sys
np_after = subprocess.run([sys.executable, "-c", "import numpy;print(numpy.__version__)"], capture_output=True, text=True).stdout.strip()
print("numpy", np_before, "->", np_after)
print("RESTART the session now (Runtime > Restart session), then run from step 4." if np_after != np_before
      else "No restart needed.")

## 4. Environment for TensorRT libraries

In [ ]:
%cd /content/pathrank
import os, glob, site, torch
# re-establish state in case the session was restarted after installs
GPU = torch.cuda.get_device_name(0)
SAVE_TO_DRIVE = os.path.exists("/content/drive/MyDrive")
print("GPU:", GPU, "| Drive mounted:", SAVE_TO_DRIVE)
# make pip-installed TensorRT libraries visible to ONNX Runtime's TensorRT provider
libdirs = [d for p in site.getsitepackages() for d in glob.glob(p + "/tensorrt_libs") + glob.glob(p + "/tensorrt_cu*_libs")]
os.environ["LD_LIBRARY_PATH"] = ":".join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")])
print("TensorRT lib dirs:", libdirs or "none found (native TensorRT still works; ORT-TensorRT may be skipped)")
import onnxruntime as ort
print("ORT", ort.__version__, ort.get_available_providers())
try:
    import tensorrt as trt; print("TensorRT", trt.__version__)
except Exception as e:
    print("TensorRT import failed:", e)

## 5. Smoke test on synthetic data (~5 min)
Proves the GPU path works end to end before spending time on the real data. These numbers are
synthetic: they validate the pipeline, not the models.

In [ ]:
!python scripts/run_pipeline.py --config configs/synthetic_small.yaml --stages all 2>&1 | grep -E "stage_|training on|rank: |FAILED|Error"
!python scripts/gpu_benchmark.py --config configs/synthetic_small.yaml --batches 1,64,1024 --iters 20 2>&1 | tail -25

## 6. Download CareerBuilder from Kaggle
Add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab's **Secrets** panel (key icon, left sidebar), or
upload `kaggle.json`. Accept the competition rules once at
https://www.kaggle.com/c/job-recommendation/rules

In [ ]:
import os, json
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception:
    from google.colab import files
    print("No Colab secrets found: upload kaggle.json")
    up = files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    open(os.path.expanduser("~/.kaggle/kaggle.json"), "wb").write(next(iter(up.values())))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
!bash scripts/download_data.sh

## 7. Full pipeline on real data (GPU training)
Validation → Feast → two-tower + LightGCN on GPU → ranker → ONNX/Triton export → notifications →
agent → drift → MLflow registry gate → reports. Set `ANTHROPIC_API_KEY` in Secrets first if you want
the agent graded by a real LLM instead of the template.

In [ ]:
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass
!python scripts/run_pipeline.py --config configs/colab_a100.yaml --stages all 2>&1 | grep -vE "Warning|warn"

## 8. GPU inference benchmark (TensorRT etc.)

In [ ]:
!python scripts/gpu_benchmark.py --config configs/colab_a100.yaml
!python tf_baseline/two_tower_keras.py --config configs/colab_a100.yaml 2>&1 | tail -2 || true
!python scripts/run_pipeline.py --config configs/colab_a100.yaml --stages report

## 9. Results

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("RESULTS.md").read()))

## 10. Save everything
Bundles results, model card, models, the Triton repository and the TensorRT engine (not the raw data).
The `.plan` engine only works on the same GPU model + TensorRT version it was built with.

In [ ]:
import shutil, time, os
from google.colab import files
stamp = time.strftime("%Y%m%d-%H%M")
out = f"/content/pathrank_results_{GPU.replace(' ', '_')}_{stamp}"
!cd /content/pathrank && zip -q -r {out}.zip RESULTS.md mlflow.db artifacts/careerbuilder/v1 -x "*.parquet" "artifacts/careerbuilder/v1/feature_repo/*"
print("bundle:", out + ".zip", round(os.path.getsize(out + ".zip") / 2**20, 1), "MB")
if SAVE_TO_DRIVE:
    shutil.copy(out + ".zip", "/content/drive/MyDrive/")
    print("copied to Google Drive")
files.download(out + ".zip")